
# Coarse camera mapping

Sequentially displays linear phases on the SLM and detects the position of the resulting
focal spots on the camera, fitting an partial affine transform (translation, scale, and
rotation) between the camera coordinates and the coordinates of the simulated output
plane. Robust to aberrations and works with the zeroth order on or off the sensor.
Figures out any flips or rotations of the camera relative to the SLM.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from hologradpy.hardware import (
    SimulatedSLMTorch,
    SimulatedCameraTorch,
    open_camera,
    open_slm,
)

from hologradpy.calibration.camera_mapping import (
    CoarseMapper,
    CameraMapperVisualizer,
    CoarseMapperVisualizer,
)

from hologradpy.optics.systems import SLMFFT, SLMFFTAffine
from hologradpy.optics.modules.slm_fields import PixelwiseSLMField
from hologradpy.optics.modules.virtual_slms import VirtualSLM
from hologradpy.optics.complex_amplitude import ComplexAmplitude, FieldGeometry

from hologradpy.profiles.amplitude import gaussian_beam_intensity
from hologradpy.profiles.zernike import Zernike
from hologradpy.utils import get_device

device = get_device(verbose=True)
data_path = "../data/"
Path(data_path).mkdir(exist_ok=True)
# %matplotlib qt5

In [ ]:
slm_geometry = FieldGeometry(
    resolution=(1024, 1280),
    pixel_size=torch.tensor([12.5e-6, 12.5e-6], device=device),
    wavelength=torch.tensor(0.630e-6, device=device),
)

slm = open_slm(SimulatedSLMTorch, input_geometry=slm_geometry, bitdepth=8)

gaussian_intensity = gaussian_beam_intensity(
    *slm.get_spatial_grid(device),
    beam_radius=5e-3,
)

# Adding abberrations to the simulated beam
zernike = Zernike(
    slm_geometry.resolution,
    unit_disk_mode="fill",
    number_of_radial_orders=10,
    device=device,
)
coefficients = torch.rand(zernike.number_of_zernikes, device=device) * 1
zernike_phase = zernike.get_phase(coefficients)

plt.figure()
plt.imshow(zernike_phase.cpu(), cmap="magma")
plt.colorbar()
plt.title("Injected Zernike Aberrations")

aberrated_beam = ComplexAmplitude(
    gaussian_intensity.sqrt() * torch.exp(1j * zernike_phase),
    wavelength=slm_geometry.wavelength,
    pixel_size=slm_geometry.pixel_size,
    power=1e-3,
)
gaussian_beam = ComplexAmplitude(
    gaussian_intensity.sqrt() + 0j,
    wavelength=slm_geometry.wavelength,
    pixel_size=slm_geometry.pixel_size,
    power=1e-3,
)

simulated_camera_model = SLMFFTAffine(
    input_geometry=slm_geometry,
    virtual_slm=slm.virtual_slm,
    camera_resolution=(960, 1440),
    camera_pixel_size=(3.45e-6, 3.45e-6),
    focal_length=0.25,
    slm_field=PixelwiseSLMField(aberrated_beam),
    padded_resolution=(2048, 2048),
    camera_angle=15,
    camera_shift=(3e-3, 1e-3),        # (x, y) metres in the focal plane
)

camera = open_camera(
    SimulatedCameraTorch,
    slm_camera_model=simulated_camera_model,
    exposure_time=100e-3,
    quantum_efficiency=0.01,
    full_well_capacity=11e3,
    noise_level=4.0,
    nd_filter_optical_density=3,
    bitdepth=10,
    background_scatter_power=1e-4,
    background_scatter_grain_radius=20e-6,
)

test_image = camera.get_image()

plt.figure()
plt.imshow(test_image, cmap="turbo")
plt.title("Initial Camera Image")
plt.colorbar()

In [ ]:
slm_camera_model = SLMFFT(
    input_geometry=slm_geometry,
    virtual_slm=VirtualSLM(phase_scaling=1.0),
    slm_field=PixelwiseSLMField(gaussian_beam),
    focal_length=0.25,
    padded_resolution=(2048, 2048),
)

In [ ]:
coarse_mapper = CoarseMapper(
    slm=slm,
    camera=camera,
    slm_camera_model=slm_camera_model,
)
coarse_mapping = coarse_mapper.map_camera()

print(f"Camera rotation: {coarse_mapping.rotation_degrees:.2f} deg")
print(f"Camera mirrored: {coarse_mapping.is_mirrored}")
print(f"Camera scales: {coarse_mapping.scales}")
print(f"Reprojection RMS: {coarse_mapping.fit.reprojection_rms:.3f} px")
print(
    "Zeroth order (y, x): "
    f"({coarse_mapping.zeroth_order_position[0]:.0f}, "
    f"{coarse_mapping.zeroth_order_position[1]:.0f}) px "
    "(extrapolated, off the sensor)"
)

In [ ]:
coarse_mapping.save(data_path + "coarse_mapping.asdf")

In [ ]:
figure = CoarseMapperVisualizer(coarse_mapping.visualization_data).render()
figure = CameraMapperVisualizer(coarse_mapping).render()

detected = np.asarray(coarse_mapping.detected_points)
plt.figure()
plt.imshow(coarse_mapping.visualization_data.camera_image, cmap="turbo")
plt.plot(detected[:, 0], detected[:, 1], "wx", label="probe spots")
plt.legend()
plt.title("Probe Spots on the Camera")